# 08. Unsupervised Learning & Clustering: Profile Segmentation

Unlike supervised learning where data contains pre-labeled targets, **unsupervised learning** deals with datasets without explicit output labels. The goal is to discover underlying structures, patterns, or groupings within the data.

**Clustering** is the most prominent unsupervised learning technique, aiming to partition data into groups (*clusters*) such that objects in the same cluster are highly similar to each other, while objects in different clusters are highly distinct.

### Objectives of this Practical Work:
1. **K-Means Clustering**: Implement partition-based clustering and select the optimal number of clusters using the **Elbow Method** (WCSS/Inertia) and **Silhouette Analysis**.
2. **Hierarchical Clustering**: Apply agglomerative clustering and visualize hierarchical relationships using **dendrograms**.
3. **DBSCAN**: Implement density-based clustering to discover clusters of arbitrary shapes and identify **outlier / noise points**.
4. **Clustering Evaluation**: Use intrinsic metrics such as the **Silhouette Coefficient**, **Davies-Bouldin Index**, and **Calinski-Harabasz Score** to quantify clustering quality.
5. **Domain Application**: Segment user profiles into meaningful **Personality Archetypes** for Cupid Matchmaking recommendation.

> **Fun Fact 1**: Clustering is widely used in business for customer segmentation—grouping customers with similar purchasing habits so that marketing teams can design highly targeted campaigns!
>
> **Fun Fact 2**: Unlike K-Means, which forces every single point into a cluster, density-based algorithms like DBSCAN can flag points as "noise". This makes DBSCAN excellent for fraud detection since anomalous behavior stands out as noise.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))

from src.models.clustering import (
    CupidKMeansClusterer,
    CupidHierarchicalClusterer,
    CupidDBSCANClusterer,
    evaluate_clustering,
    compute_k_distance
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print('All clustering modules imported successfully!')

## 1. Data Preparation: Personality & Behavioral Profiles

For matchmaking and dating segmentation, we analyze users' fundamental personality dimensions:
- **Big 5 Traits**: Openness, Extraversion, Agreeableness, Conscientiousness
- **Lifestyle & Energy**: Chronotype, Spontaneity, Emotional Expressiveness, Career Ambition

Since clustering algorithms rely on geometric distance (e.g. Euclidean distance), feature standardization via `StandardScaler` is **mandatory** so that traits with larger variance do not artificially dominate distance calculations.

In [ ]:
df = pd.read_csv('../data/raw/cupid_algorithm_dataset.csv')

traits = [
    'openness', 'extraversion', 'agreeableness', 'conscientiousness',
    'chronotype', 'spontaneity', 'emotional_expressiveness', 'career_ambition'
]

# Sample 10,000 user profiles for responsive clustering and visualization
a_cols = [f'a_{t}' for t in traits]
profiles_df = df[a_cols].sample(n=10000, random_state=42).copy()
profiles_df.columns = traits

# Standardization (Z-score normalization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(profiles_df)

print(f'Profiles shape: {profiles_df.shape}')
profiles_df.head()

## 2. K-Means Clustering

K-Means is a partition-based algorithm that divides $n$ observations into $k$ clusters, where each observation belongs to the cluster with the nearest mean (centroid).

$$\arg\min_{\mathbf{S}} \sum_{i=1}^{k} \sum_{\mathbf{x} \in S_i} ||\mathbf{x} - \boldsymbol{\mu}_i||^2$$

To choose the optimal $k$, we use two complementary techniques:
1. **The Elbow Method**: Plots Within-Cluster Sum of Squares (WCSS / Inertia) against $k$. We look for the "elbow point" where the rate of decrease abruptly slows.
2. **Silhouette Analysis**: Measures how similar an object is to its own cluster compared to other clusters ($s \in [-1, 1]$).

In [ ]:
k_range = list(range(2, 9))
inertias = []
silhouette_scores = []

for k in k_range:
    km = CupidKMeansClusterer(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    metrics = evaluate_clustering(X_scaled, km.labels_, sample_size=3000, random_state=42)
    silhouette_scores.append(metrics['Silhouette_Score'])
    print(f'k = {k:2d} | WCSS (Inertia): {km.inertia_:10.2f} | Silhouette Score: {metrics["Silhouette_Score"]:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_title('Elbow Method (WCSS / Inertia)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)', fontsize=11)
ax1.set_ylabel('Inertia (WCSS)', fontsize=11)
ax1.set_xticks(k_range)
ax1.grid(True)

ax2.plot(k_range, silhouette_scores, 'rs-', linewidth=2, markersize=8)
ax2.set_title('Silhouette Analysis across k', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)', fontsize=11)
ax2.set_ylabel('Average Silhouette Coefficient', fontsize=11)
ax2.set_xticks(k_range)
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
optimal_k = 4
best_kmeans = CupidKMeansClusterer(n_clusters=optimal_k, random_state=42)
kmeans_labels = best_kmeans.fit_predict(X_scaled)

kmeans_eval = evaluate_clustering(X_scaled, kmeans_labels, f'K-Means (k={optimal_k})')
print(f'=== K-Means Performance (k={optimal_k}) ===')
for metric, val in kmeans_eval.items():
    print(f'{metric:25s}: {val}')

## 3. Hierarchical Agglomerative Clustering & Dendrogram

Hierarchical clustering builds a bottom-up tree of clusters (*agglomerative*). Every profile starts as its own cluster, and at each step, the two closest clusters are merged based on a linkage criterion:
- **Ward Linkage**: Minimizes the total within-cluster variance.
- **Complete Linkage**: Uses the maximum distance between points in two clusters.
- **Average Linkage**: Uses the average distance between all pairs.

The **Dendrogram** provides a complete hierarchical view. Cutting the dendrogram horizontally at a given dissimilarity threshold yields the desired partition.

In [ ]:
# Subsample for clean dendrogram visualization
dendro_sample = X_scaled[:150]
linkage_matrix = CupidHierarchicalClusterer.compute_linkage_matrix(dendro_sample, method='ward')

plt.figure(figsize=(14, 6))
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontsize=14, fontweight='bold')
plt.xlabel('Sample Index / Cluster Size', fontsize=11)
plt.ylabel('Euclidean Distance', fontsize=11)
dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=25,
    leaf_rotation=90.,
    leaf_font_size=10.,
    show_contracted=True
)
plt.axhline(y=14.0, color='r', linestyle='--', linewidth=2, label='Threshold Cut (Yields k=4 clusters)')
plt.legend(fontsize=11)
plt.show()

In [ ]:
# Fit Hierarchical on a 4,000 profile sample
hier_sample_size = 4000
X_hier = X_scaled[:hier_sample_size]
hierarchical = CupidHierarchicalClusterer(n_clusters=optimal_k, linkage_method='ward')
hier_labels = hierarchical.fit_predict(X_hier)

hier_eval = evaluate_clustering(X_hier, hier_labels, f'Hierarchical Ward (k={optimal_k})')
print('=== Hierarchical Clustering Performance ===')
for metric, val in hier_eval.items():
    print(f'{metric:25s}: {val}')

## 4. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

DBSCAN is fundamentally different from K-Means and Hierarchical clustering:
1. **Arbitrary Shapes**: It groups points that are closely packed together (points with many nearby neighbors) and can discover clusters of arbitrary, non-spherical shapes.
2. **Outlier / Noise Detection**: Points that lie alone in low-density regions are flagged as **noise** (label = `-1`).

### Hyperparameters:
- `eps` ($\epsilon$): The maximum distance between two samples for one to be considered in the neighborhood of the other.
- `min_samples`: The number of samples in a neighborhood for a point to be considered a **core point**.

To choose $\epsilon$, we plot the distance to the $k$-th nearest neighbor ($k = \text{min\_samples}$) sorted in ascending order and search for the **knee point**.

In [ ]:
k_dist = compute_k_distance(X_scaled[:4000], k=5)

plt.figure(figsize=(9, 4.5))
plt.plot(k_dist, color='#2c3e50', linewidth=2)
plt.axhline(y=1.85, color='#e67e22', linestyle='--', linewidth=2, label='Chosen Epsilon (eps = 1.85)')
plt.title('k-Distance Graph for DBSCAN Epsilon (eps) Selection', fontsize=13, fontweight='bold')
plt.xlabel('Points Sorted by Distance to 5th Nearest Neighbor', fontsize=11)
plt.ylabel('5-NN Distance', fontsize=11)
plt.legend(fontsize=11)
plt.grid(True)
plt.show()

# Fit DBSCAN
dbscan = CupidDBSCANClusterer(eps=1.85, min_samples=25)
dbscan_labels = dbscan.fit_predict(X_scaled)
dbscan_eval = evaluate_clustering(X_scaled, dbscan_labels, f'DBSCAN (eps={dbscan.eps})')

print(f'Dense clusters found : {dbscan.n_clusters_}')
print(f'Noise / outlier points : {dbscan.n_noise_} ({dbscan.n_noise_ / len(X_scaled) * 100:.2f}%)')
print(f'Silhouette (non-noise): {dbscan_eval["Silhouette_Score"]}')

## 5. Clustering Evaluation: Intrinsic Metrics Benchmark

Unlike supervised learning, there are no ground-truth labels. We evaluate clustering using **intrinsic quality metrics**:

| Metric | Formula / Principle | Optimal Direction |
|---|---|---|
| **Silhouette Coefficient** | $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$, measures cohesion vs separation in $[-1, 1]$ | **Higher is better** (closer to 1) |
| **Davies-Bouldin Index** | Ratio of within-cluster scatter to between-cluster separation | **Lower is better** (closer to 0) |
| **Calinski-Harabasz Score** | Ratio of between-cluster dispersion to within-cluster dispersion (Variance Ratio) | **Higher is better** |

In [ ]:
comparison_df = pd.DataFrame([kmeans_eval, hier_eval, dbscan_eval])
display(comparison_df)

## 6. PCA 2D Visualization & Personality Archetypes

We project our 8-dimensional personality space onto 2 principal components (PCA) to visualize cluster boundaries, centroids, and DBSCAN noise points.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var1, var2 = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# 1. K-Means
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='viridis', alpha=0.4, s=15)
centroids_pca = pca.transform(best_kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='red', marker='X', s=200, edgecolors='black', label='Centroids')
axes[0].set_title(f'K-Means (k={optimal_k})', fontsize=13, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({var1*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({var2*100:.1f}%)')
axes[0].legend()
axes[0].grid(True)

# 2. Hierarchical
axes[1].scatter(X_pca[:hier_sample_size, 0], X_pca[:hier_sample_size, 1], c=hier_labels, cmap='tab10', alpha=0.4, s=15)
axes[1].set_title(f'Hierarchical Ward (k={optimal_k})', fontsize=13, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({var1*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({var2*100:.1f}%)')
axes[1].grid(True)

# 3. DBSCAN
import matplotlib.colors as mcolors
colors = ['#95a5a6' if l == -1 else mcolors.to_hex(plt.cm.tab10(l % 10)) for l in dbscan_labels]
axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.4, s=15)
axes[2].set_title(f'DBSCAN (eps={dbscan.eps})\nGrey: Outliers ({dbscan.n_noise_})', fontsize=13, fontweight='bold')
axes[2].set_xlabel(f'PC1 ({var1*100:.1f}%)')
axes[2].set_ylabel(f'PC2 ({var2*100:.1f}%)')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 7. Business & Domain Interpretation: Personality Archetypes

In matchmaking platforms like Cupid, customer segmentation allows the algorithm to quickly identify personality archetypes and suggest compatible pairs across different clusters.

In [ ]:
profiles_df['Cluster'] = kmeans_labels
archetypes = profiles_df.groupby('Cluster')[traits].mean().round(2)
sizes = profiles_df['Cluster'].value_counts().sort_index()
archetypes['User_Count'] = sizes
archetypes['Percentage'] = (sizes / len(profiles_df) * 100).round(1)

labels_map = {
    0: 'The Grounded Stabilizers (High Agreeableness & Conscientiousness)',
    1: 'The Dynamic Visionaries (High Openness & Ambition)',
    2: 'The Spontaneous Socializers (High Extraversion & Spontaneity)',
    3: 'The Independent Thinkers (Low Extraversion, High Conscientiousness)'
}
archetypes['Archetype_Name'] = [labels_map.get(i, f'Archetype {i}') for i in archetypes.index]

display(archetypes[['Archetype_Name', 'User_Count', 'Percentage'] + traits])

## 8. Summary of Findings & Takeaways

- **K-Means**: Highly efficient ($O(N)$), fast convergence, perfectly partitions user profiles into 4 distinct, balanced personality archetypes.
- **Hierarchical Clustering**: Ward linkage generates clear dendrogram trees showing how individual profiles merge into broad temperaments, confirming the natural 4-cluster structure.
- **DBSCAN**: Effectively detects dense personality clusters while isolating unusual, atypical profile combinations as noise points (useful for bot detection or edge-case matching).
- **Evaluation Metrics**: Silhouette analysis, Davies-Bouldin, and Calinski-Harabasz confirm that $k=4$ provides the optimal trade-off between cluster separation and profile cohesion.